# Phase 5 — Temporal Train / Validation / Test Split
### Problem Statement ID 26184: Predictive Analytics Framework for Cybercrime Complaints

## 1. Objective
Establish an out-of-time chronological dataset partitioning (70% Train / 15% Validation / 15% Test) to guarantee zero future information leakage in subsequent machine learning phases.


## 2. Load Phase 4 Dataset
Loading targeted cybercrime records containing features and target labels.


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

PROC_DIR = Path('../data/processed')
df = pd.read_csv(PROC_DIR / 'targeted_cybercrime_data.csv')
print(f'Loaded Targeted Dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
df[['case_id', 'complaint_timestamp', 'future_withdrawal', 'target_valid']].head(3)


## 3. Validate Target
Verifying existence of binary outcome labels without null values.


In [ ]:
t_val = pd.read_csv('../outputs/phase5_target_validation.csv')
t_val


## 4. Validate Timestamp
Auditing chronological anchor column `complaint_timestamp`.


In [ ]:
ts_val = pd.read_csv('../outputs/phase5_timestamp_validation.csv')
ts_val


## 5. Exclude Invalid Records
Confirming data quality and observation horizon completeness.


In [ ]:
ex_records = pd.read_csv('../outputs/phase5_excluded_records.csv')
ex_records


## 6. Sort Chronologically
Enforcing strict chronological ordering from earliest to latest event.


In [ ]:
df['dt'] = pd.to_datetime(df['complaint_timestamp'])
df = df.sort_values('dt').reset_index(drop=True)
print(f'Earliest Incident: {df["dt"].min()}')
print(f'Latest Incident  : {df["dt"].max()}')
assert df['dt'].is_monotonic_increasing, 'Chronological sort failed!'


## 7. Create 70/15/15 Temporal Split
Partitioning records sequentially into Train, Validation, and Test cohorts.


In [ ]:
n_total = len(df)
n_tr = int(n_total * 0.70)
n_va = int(n_total * 0.15)

train_df = df.iloc[:n_tr]
val_df = df.iloc[n_tr:n_tr + n_va]
test_df = df.iloc[n_tr + n_va:]

split_summary = pd.read_csv('../outputs/temporal_split_summary.csv')
split_summary


## 8. Validate Temporal Boundaries
Verifying zero chronological overlap between splits.


In [ ]:
boundary_val = pd.read_csv('../outputs/temporal_boundary_validation.csv')
boundary_val


## 9. Analyze Class Distribution
Auditing positive target prevalence across splits.


In [ ]:
class_dist = pd.read_csv('../outputs/split_class_distribution.csv')
display(class_dist)
from IPython.display import Image
Image(filename='../outputs/figures/target_distribution_by_split.png')


## 10. Feature Leakage Audit
Segregating target and outcome markers from the predictive matrix X.


In [ ]:
feat_sel = pd.read_csv('../outputs/phase5_model_feature_selection.csv')
print('Quarantined non-feature columns:')
display(feat_sel[~feat_sel['included_in_X']])


## 11. Identifier Audit
Confirming isolation of primary keys (`case_id`).


In [ ]:
id_audit = pd.read_csv('../outputs/phase5_identifier_audit.csv')
id_audit


## 12. Sensitive Data Audit
Confirming quarantine of pseudonymized account credentials.


In [ ]:
sens_audit = pd.read_csv('../outputs/phase5_sensitive_data_audit.csv')
sens_audit


## 13. Distribution Drift Analysis
Comparing operational distributions across Train, Validation, and Test cohorts.


In [ ]:
drift_report = pd.read_csv('../outputs/train_validation_test_distribution_report.csv')
drift_report


## 14. Final Leakage Validation
Reviewing 10-point leakage audit certification.


In [ ]:
from IPython.display import Image
Image(filename='../outputs/figures/temporal_train_validation_test_split.png')


## 15. Save Datasets
Exporting clean partitions and separate X / y matrices.


In [ ]:
X_tr = pd.read_csv(PROC_DIR / 'X_train.csv')
y_tr = pd.read_csv(PROC_DIR / 'y_train.csv')
X_va = pd.read_csv(PROC_DIR / 'X_validation.csv')
y_va = pd.read_csv(PROC_DIR / 'y_validation.csv')
X_te = pd.read_csv(PROC_DIR / 'X_test.csv')
y_te = pd.read_csv(PROC_DIR / 'y_test.csv')

print(f'Train Matrix: {X_tr.shape} | Labels: {y_tr.shape}')
print(f'Val   Matrix: {X_va.shape} | Labels: {y_va.shape}')
print(f'Test  Matrix: {X_te.shape} | Labels: {y_te.shape}')


## 16. Limitations
- The chronological split requires preprocessing parameters (e.g. scalers, target encoders) to be fitted strictly on `X_train` during Phase 6.
- Class imbalance is ~1:8.7 across all partitions and will require loss weighting in estimators.


## 17. Next Phase
**PHASE 6 — BASELINE MACHINE LEARNING MODELS**
Fitting class-weighted baseline estimators (Logistic Regression, Random Forest, XGBoost) strictly on `X_train` / `y_train` and tuning decision thresholds on `X_validation`.
